# Istraživanje podataka — Klasifikacija support tiketa

U ovom notebooku vršimo **exploratory data analysis (EDA)** nad skupom podataka za klasifikaciju korisničkih support tiketa.

**Cilj:** Razumeti strukturu dataseta, distribuciju klasa, dužinu tekstova i potencijalnu nebalansiranost pre treniranja neuronske mreže.

**Izvor podataka:** Javni HuggingFace dataset [`Tobi-Bueck/customer-support-tickets`](https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets).

> **Napomena:** Originalno planirani dataseti (`Shekswess/zendesk_tickets_multi_class` i `dariast/support_tickets_classification`) nisu bili javno dostupni na HuggingFace-u, pa koristimo ovaj zamenski dataset sa istom domenom (subject, body, queue).

**Labela za klasifikaciju:** Kolona `queue` označava departman kojem tiket pripada (npr. *Technical Support*, *Billing and Payments*).

## 1. Priprema okruženja

Učitavamo potrebne biblioteke i module projekta. Putanja do korena projekta se dodaje u `sys.path` kako bismo mogli da importujemo `src.config` i `src.data_loader`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from wordcloud import STOPWORDS, WordCloud

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.data_loader import (
    compute_imbalance_metrics,
    get_basic_info,
    get_class_distribution,
    get_examples_per_class,
    get_text_length_stats,
    load_raw_dataset,
    to_dataframe,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Učitavanje dataseta

Dataset se učitava pomoću HuggingFace `datasets` biblioteke. Tekst tiketa se formira spajanjem kolona `subject` i `body`, a kategorija se uzima iz kolone `queue`.

Za analizu filtriramo samo **engleske** tikete (`language == "en"`), jer će model `distilbert-base-uncased` raditi na engleskom jeziku.

In [ ]:
raw_dataset = load_raw_dataset()
df = to_dataframe(raw_dataset, language=config.LANGUAGE_FILTER)

print(f"Učitano primera: {len(df):,}")
print(f"Sačuvano u: {config.RAW_DATA_FILE}")
df.head()

## 3. Osnovne statistike

Pregledamo broj primera, broj klasa i dostupne kolone u normalizovanom DataFrame-u.

In [ ]:
info = get_basic_info(df)
print(f"Broj primera: {info['num_examples']:,}")
print(f"Broj klasa: {info['num_classes']}")
print(f"Kategorije: {info['categories']}")
print(f"\nKolone: {info['columns']}")

## 4. Distribucija klasa

Brojimo koliko tiketa pripada svakoj kategoriji (`queue`). Ova distribucija je ključna za razumevanje zadatka klasifikacije i potencijalnih problema sa nebalansiranim klasama.

In [ ]:
class_dist = get_class_distribution(df)
class_pct = (class_dist / len(df) * 100).round(2)

dist_table = pd.DataFrame({
    "broj_primera": class_dist,
    "procenat (%)": class_pct,
})
dist_table

## 5. Statistike dužine teksta

Analiziramo dužinu tiketa u **karakterima** i **rečima**. Ovo pomaže pri izboru `MAX_SEQUENCE_LENGTH` za tokenizaciju (trenutno 128 u `config.py`).

In [ ]:
length_stats = get_text_length_stats(df)
length_stats

In [ ]:
df["char_length"] = df[config.TEXT_COLUMN].str.len()
df["word_length"] = df[config.TEXT_COLUMN].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["char_length"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribucija dužine teksta (karakteri)")
axes[0].set_xlabel("Broj karaktera")

sns.histplot(df["word_length"], bins=50, kde=True, ax=axes[1], color="coral")
axes[1].set_title("Distribucija dužine teksta (reči)")
axes[1].set_xlabel("Broj reči")

plt.tight_layout()
plt.show()

## 6. Vizualizacija distribucije klasa (bar chart)

Stubičasti dijagram prikazuje broj tiketa po kategoriji. Vizuelno je lako uočiti koje klase imaju više, a koje manje primera.

In [ ]:
plot_df = class_dist.reset_index()
plot_df.columns = ["category", "count"]

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="category", y="count", hue="category", palette="viridis", legend=False)
plt.title("Distribucija support tiketa po kategorijama")
plt.xlabel("Kategorija (queue)")
plt.ylabel("Broj tiketa")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Wordcloud — najčešće reči

Wordcloud vizualizacija prikazuje najfrekventnije reči u tekstovima tiketa. Prvo prikazujemo **globalni** wordcloud za ceo skup, zatim po jedan za svaku kategoriju.

In [ ]:
all_text = " ".join(df[config.TEXT_COLUMN].tolist())

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    stopwords=STOPWORDS,
    colormap="viridis",
).generate(all_text)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Wordcloud — svi engleski tiketi")
plt.show()

In [ ]:
categories = sorted(df[config.LABEL_COLUMN].unique())
n_cols = 3
n_rows = (len(categories) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for idx, category in enumerate(categories):
    text = " ".join(df[df[config.LABEL_COLUMN] == category][config.TEXT_COLUMN].tolist())
    wc = WordCloud(
        width=400,
        height=300,
        background_color="white",
        stopwords=STOPWORDS,
    ).generate(text)
    axes[idx].imshow(wc, interpolation="bilinear")
    axes[idx].set_title(category, fontsize=10)
    axes[idx].axis("off")

for idx in range(len(categories), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Wordcloud po kategorijama", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Detekcija nebalansiranosti klasa

Nebalansirani dataset može dovesti do toga da model favorizuje klase sa više primera. Računamo odnos najveće i najmanje klase (**imbalance ratio**). Ako je odnos veći od 1.5, smatramo da je dataset nebalansiran.

In [ ]:
imbalance = compute_imbalance_metrics(df)

print("=== Analiza nebalansiranosti ===")
print(f"Najveća klasa: {imbalance['max_class']} ({imbalance['max_count']} primera, {imbalance['max_class_pct']}%)")
print(f"Najmanja klasa: {imbalance['min_class']} ({imbalance['min_count']} primera, {imbalance['min_class_pct']}%)")
print(f"Imbalance ratio (max/min): {imbalance['imbalance_ratio']}")
print(f"Dataset je nebalansiran: {imbalance['is_imbalanced']}")

if imbalance["is_imbalanced"]:
    print("\nPreporuka: Razmotriti tehnike balansiranja (npr. class weights, oversampling ili undersampling) tokom treniranja.")
else:
    print("\nDistribucija klasa je relativno uravnotežena za treniranje klasifikatora.")

## 9. Primeri tiketa iz svake kategorije

Pregledamo po jedan reprezentativan tiket iz svake kategorije kako bismo intuitivno razumeli sadržaj i razliku između klasa.

In [ ]:
examples = get_examples_per_class(df, n=1)

for category, texts in examples.items():
    print("=" * 80)
    print(f"Kategorija: {category}")
    print("-" * 80)
    print(texts[0][:800] + ("..." if len(texts[0]) > 800 else ""))
    print()

## Zaključak

U ovom notebooku smo:
1. Učitali dataset sa HuggingFace-a i normalizovali ga u kolone `text` i `category`
2. Prikazali osnovne statistike i distribuciju klasa
3. Analizirali dužinu tekstova tiketa
4. Vizualizovali distribuciju klasa i wordcloud
5. Detektovali nebalansiranost klasa
6. Pregledali primere tiketa iz svake kategorije

Sledeći korak: priprema podataka za treniranje (train/val/test split) i fine-tuning DistilBERT modela.